# Bot Standards

Standards every Mad House bot must follow. These exist so any bot can be picked up, debugged, and handed off without a walkthrough.

## Naming and Identity

- Bot names are lowercase, one word or hyphenated: `chopsticks`, `snek`, `hank-duck`
- GitHub repo: same name as the bot
- Docker container name in Coolify: same as the bot
- Discord application name: display name (can differ from repo name)

**Do not** name a bot after its tech stack or its purpose. Name it like a character.

## Command Structure

All commands are slash commands. No prefix commands in new bots.

| Rule | Reason |
|------|--------|
| Slash commands only | Discord is deprecating message prefix bots |
| `ephemeral=True` for admin commands | Keeps channels clean |
| `ephemeral=True` for error responses | Errors are between the user and the bot |
| Deferred responses for long ops | Discord gives only 3 seconds before timeout |
| Cooldowns on expensive commands | Prevents abuse |

### Deferred response pattern

```python
@app_commands.command(name="generate")
async def generate(self, interaction: discord.Interaction, prompt: str):
    await interaction.response.defer()  # Extends timeout to 15 minutes
    result = await self.do_long_operation(prompt)
    await interaction.followup.send(result)
```

## Required Files

Every bot repo must have:

```
README.md              What it does, how to run it locally, env vars needed
.env.example           All required env vars with placeholder values
Dockerfile             Always. Even for simple bots.
requirements.txt       Pinned versions
.gitignore             .env, __pycache__, *.pyc, venv/
CHANGELOG.md           Updated on every release
```

**Never commit:** `.env`, secrets, `node_modules/`, `venv/`, `__pycache__/`

## Logging Standards

```python
import logging

# Format: timestamp level name: message
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s"
)

log = logging.getLogger(__name__)

# What to log at each level:
# DEBUG: verbose internals, only for local dev
# INFO:  lifecycle events (startup, shutdown, user action completed)
# WARNING: recoverable issues (rate limit hit, missing optional config)
# ERROR: failed operations (DB write failed, API call failed)
```

**Never log:** tokens, passwords, full message content, user PII.

## Security Checklist

Before marking any bot production-ready:

- [ ] Bot token loaded from env, never hardcoded
- [ ] Database credentials loaded from env
- [ ] User input sanitized before use in SQL or shell commands
- [ ] No `eval()` or `exec()` on user-provided strings
- [ ] Rate limits on all commands that call external APIs
- [ ] Error messages never expose internal state or stack traces
- [ ] Dockerfile does not run as root
- [ ] .env not in `.gitignore` exceptions
- [ ] Required intents declared and no extras requested
- [ ] Admin commands check for required permissions explicitly

## Patterns: What Works

### Config via Pydantic Settings

```python
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    token: str
    database_url: str
    redis_url: str = "redis://localhost:6379"
    dev_guild_id: int | None = None
    cogs: list[str] = ["bot.cogs.admin", "bot.cogs.fun"]

    class Config:
        env_file = ".env"

settings = Settings()
```

### Cog pattern

```python
class MyCog(commands.Cog):
    def __init__(self, bot: commands.Bot):
        self.bot = bot
        # Access bot.pool, bot.redis, etc. here

async def setup(bot: commands.Bot):
    await bot.add_cog(MyCog(bot))
```

### Graceful shutdown

```python
async def close(self):
    # Close DB pool, redis, etc. before shutdown
    if hasattr(self, 'pool'):
        await self.pool.close()
    await super().close()
```

## Handoff Template

When handing off a bot to another developer or agent, include:

```markdown
# Bot: [name]

**Purpose:** One sentence.

**Status:** [development / staging / production]

**Deployment:** Coolify project [name], container [name], branch [name]

**Dependencies:** Postgres [version], Redis [version], external APIs:
- [API name]: used for [purpose], key stored as [ENV_VAR_NAME]

**Known issues:**
- [issue description]

**Environment variables:** see .env.example. All required vars must be set in Coolify.

**How to run locally:** `docker compose up`

**Last deployer:** [name, date]
```

## Pre-Launch Checklist

Run through this before inviting the bot to a production server:

- [ ] All commands tested in a dev server with real users
- [ ] Error handling tested: force an error and verify the user sees a clean message
- [ ] Long-running commands tested with deferred response
- [ ] Bot runs for 24h in staging without crashes
- [ ] Docker logs clean: no recurring ERROR or WARNING lines
- [ ] Memory and CPU usage stable (not leaking)
- [ ] Rollback plan confirmed: previous image tagged, can re-deploy in under 2 min
- [ ] Bot permissions reviewed: nothing excessive
- [ ] README up to date